#Reconciling the three data sources: Excel labels, crops, originals

In [ ]:
# Paths: set DATA_DIR / OUTPUT_DIR as environment variables, or edit the defaults below.
import os
DATA_DIR = os.environ.get("DATA_DIR", "../data")


In [2]:
pass  # Colab only: from google.colab import drive
from pathlib import Path

pass  # Colab only: drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from pathlib import Path

ROOT = Path(DATA_DIR)
XLSX = ROOT / '00_raw' / 'annotations.xlsx'

In [3]:
df = pd.read_excel(XLSX)
df.columns = [str(c).strip() for c in df.columns]

MAT_COL = 'Material'
print("Columns in sheet:", list(df.columns), "\n")

counts = df[MAT_COL].fillna('(blank)').astype(str).str.strip().value_counts()
print(f"{len(counts)} distinct raw material values:\n")
for val, n in counts.items():
    print(f"{n:4}  |{val}|")

Columns in sheet: ['Inv. no.', 'Object', 'Material'] 

106 distinct raw material values:

 177  |Green glass|
 118  |Glass|
  28  |Earthenware|
  28  |Metal alloy|
  17  |Greenish glass|
  16  |Blue and white porcelain|
  13  |Wood|
  10  |(blank)|
  10  |Earthen ware|
   9  |Porcelain|
   8  |Celadon|
   7  |Stone|
   6  |Brownish glass|
   5  |Enameled glass|
   5  |Bronze|
   5  |Bone|
   5  |Sandstone|
   4  |Iron|
   4  |Stoneware|
   4  |Haysi Ware|
   3  |Olive green glass|
   3  |Brown glass|
   3  |Ceramics|
   3  |Ivory|
   3  |Clay|
   3  |Green Glass|
   3  |Black glass|
   3  |Bahla Ware|
   3  |Greyish glass|
   3  |Stone (diorite)|
   3  |Stone (granite)|
   2  |Whitish glass|
   2  |Glaze pottery|
   2  |Ceramic|
   2  |Qingbai porcelain or celadon|
   2  |Sulphur|
   2  |Julfar Ware|
   2  |Grey glass|
   2  |Fritware|
   2  |Blue and White porcelain|
   2  |Limestone|
   2  |Dot and Circle Ware|
   1  |Soft-stone|
   1  |Earthenware with buff wash|
   1  |Earthenware 

In [4]:
df['Material'] = df['Material'].fillna('(blank)').astype(str).str.strip()

MATERIAL_MAP = {
    # ---- GLASS ----
    'Green glass':'glass','Glass':'glass','Greenish glass':'glass','Brownish glass':'glass',
    'Enameled glass':'glass','Olive green glass':'glass','Brown glass':'glass','Green Glass':'glass',
    'Black glass':'glass','Greyish glass':'glass','Whitish glass':'glass','Grey glass':'glass',
    'Orange glass':'glass','-Glass':'glass','Greysh glass':'glass','Bluish glass':'glass',
    'Brownish/Yellowish Glass':'glass','Black and white glass':'glass','Yellowish glass':'glass',
    'Blueish glass':'glass',
    # ---- METAL ----
    'Metal alloy':'metal','Bronze':'metal','Iron':'metal','Silver':'metal',
    # ---- POTTERY (ceramic family: earthenware, porcelain, celadon, stoneware, wares, frit, clay) ----
    'Earthenware':'pottery','Blue and white porcelain':'pottery','Earthen ware':'pottery',
    'Porcelain':'pottery','Celadon':'pottery','Stoneware':'pottery','Haysi Ware':'pottery',
    'Ceramics':'pottery','Bahla Ware':'pottery','Glaze pottery':'pottery','Ceramic':'pottery',
    'Qingbai porcelain or celadon':'pottery','Julfar Ware':'pottery','Fritware':'pottery',
    'Blue and White porcelain':'pottery','Dot and Circle Ware':'pottery',
    'Earthenware with buff wash':'pottery','Earthenware with numerous white inclusions':'pottery',
    'Earthenware; smoothing':'pottery','Bahla ware':'pottery','Red grit ware':'pottery',
    'Abbasid barbotine (pottery)':'pottery','Indian Red Ware':'pottery','Pottery':'pottery',
    'Earthenware, covered by buff wash':'pottery','Monochrome turquoise glaze pottery':'pottery',
    'Earthenware with dark red slip':'pottery','Clay':'pottery','Eartheware':'pottery',
    'Erathenware':'pottery','Earthenware (Fine white cream fabric)':'pottery','Martaban':'pottery',
    'Eathernware':'pottery','Chinese Porcelain':'pottery','Glaze ware':'pottery',
    'Longquan celadon':'pottery','Stoneware Martaban':'pottery','Friitware':'pottery',
    'Martaban ware':'pottery','Polycromous ware (Porcelain)':'pottery','White porcelain':'pottery',
    'Stoneware/ celadon':'pottery','Batavia ware':'pottery','White procelain':'pottery',
    'Blue and white procelain':'pottery','Polychromous ware (porcelain)':'pottery',
    'Stoneware (celadon)':'pottery','Dehua ware':'pottery','Polychromous ware':'pottery',
    'Frit ware':'pottery','Eggshell':'pottery','Vitrified clay':'pottery',
    'Vetrified clay with metal inside':'pottery',
    'Earthenware with buff wash and dark red slip in the centre of each wall':'pottery',
    'Islamic glaze':'pottery','Glaze':'pottery','sphero conical vessel':'pottery',
    # ---- MISC (stone, bone, wood, organics, other substances) ----
    'Wood':'misc','Stone':'misc','Sandstone':'misc','Bone':'misc','Ivory':'misc',
    'Stone (diorite)':'misc','Stone (granite)':'misc','Sulphur':'misc','Limestone':'misc',
    'Soft-stone':'misc','Fibre (coconut or date)':'misc',
    'Stone (hematite for production of red paint)':'misc','Shell':'misc','Volcanic stone':'misc',
    'Copal':'misc','Wood or bone':'misc','Bone/Ivory':'misc','Coral':'misc',
    'Stone (obsidian)':'misc','Softstone':'misc','Stone (flint)':'misc','Stone (Softstone)':'misc',
    'stone':'misc','Basalt':'misc',
    # ---- UNLABELLED ----
    '(blank)':'unlabelled',
}

df['Material_4'] = df['Material'].map(MATERIAL_MAP)

unmapped = df[df['Material_4'].isna()]['Material'].unique()
if len(unmapped):
    print("UNMAPPED values (fix the dictionary):")
    for v in unmapped: print(f"   |{v}|")
else:
    print("Every material value mapped.\n")

print("Harmonised distribution:")
print(df['Material_4'].value_counts(dropna=False).to_string())

Every material value mapped.

Harmonised distribution:
Material_4
glass         350
pottery       142
misc           58
metal          38
unlabelled     10


In [6]:
# Building working driectory: usable-only crops + originals + clean Excel
import shutil
from pathlib import Path

ROOT     = Path(DATA_DIR)
RAW      = ROOT / '00_raw'
WORKING  = ROOT / '00_working'
(WORKING / 'crops').mkdir(parents=True, exist_ok=True)
(WORKING / 'originals').mkdir(parents=True, exist_ok=True)

IMG_EXT = {'.jpg', '.jpeg', '.png'}
def ids_from_dir(d):
    return {p.stem.strip() for p in d.iterdir() if p.suffix.lower() in IMG_EXT}

# --- the three ID sets ---
excel_ids = set(df['Inv. no.'].dropna().astype(str).str.strip())
crop_ids  = ids_from_dir(RAW / 'crops')
orig_ids  = ids_from_dir(RAW / 'originals')

# --- usable = present in all three ---
usable = excel_ids & crop_ids & orig_ids

print(f"Excel: {len(excel_ids)} | Crops: {len(crop_ids)} | Originals: {len(orig_ids)}")
print(f"USABLE (all three): {len(usable)}\n")
print("Excluded / mismatches:")
print(f"  Excel rows with no crop     : {len(excel_ids - crop_ids)}")
print(f"  Excel rows with no original : {len(excel_ids - orig_ids)}")
print(f"  Crops with no Excel row     : {len(crop_ids - excel_ids)}")
print(f"  Crops with no original      : {len(crop_ids - orig_ids)}")
print(f"  Originals with no Excel row : {len(orig_ids - excel_ids)}")
print(f"  Originals with no crop      : {len(orig_ids - crop_ids)}")

# --- copying usable crops + originals into 00_working/ ---
def copy_usable(src_dir, dst_dir):
    n = 0
    for p in src_dir.iterdir():
        if p.stem.strip() in usable and p.suffix.lower() in IMG_EXT:
            shutil.copy2(p, dst_dir / p.name); n += 1
    return n

nc = copy_usable(RAW/'crops',     WORKING/'crops')
no = copy_usable(RAW/'originals', WORKING/'originals')
print(f"\nCopied {nc} crops, {no} originals into 00_working/")

# --- clean Excel Sheet ---
clean = df[df['Inv. no.'].astype(str).str.strip().isin(usable)].copy()
clean.to_excel(WORKING / 'annotations_clean.xlsx', index=False)
print(f"Wrote annotations_clean.xlsx: {len(clean)} rows")

(RAW / 'usable_ids.txt').write_text("\n".join(sorted(usable)))

print("\nWorking-set material distribution:")
print(clean['Material_4'].value_counts(dropna=False).to_string())

Excel: 598 | Crops: 581 | Originals: 578
USABLE (all three): 559

Excluded / mismatches:
  Excel rows with no crop     : 39
  Excel rows with no original : 33
  Crops with no Excel row     : 22
  Crops with no original      : 9
  Originals with no Excel row : 13
  Originals with no crop      : 6

Copied 559 crops, 559 originals into 00_working/
Wrote annotations_clean.xlsx: 559 rows

Working-set material distribution:
Material_4
glass         332
pottery       131
misc           51
metal          37
unlabelled      8
